# Softmax Regression
:label:`sec_softmax`

In :numref:`sec_linear_regression`, we introduced linear regression,
working through implementations from scratch in :numref:`sec_linear_scratch`
and again using high-level APIs of a deep learning framework
in :numref:`sec_linear_concise` to do the heavy lifting.

Regression is the hammer we reach for when
we want to answer *how much?* or *how many?* questions.
If you want to predict the number of dollars (price)
at which a house will be sold,
or the number of wins a baseball team might have,
or the number of days that a patient
will remain hospitalized before being discharged,
then you are probably looking for a regression model.
However, even within regression models,
there are important distinctions.
For instance, the price of a house
will never be negative and changes might often be *relative* to its baseline price.
As such, it might be more effective to regress
on the logarithm of the price.
Likewise, the number of days a patient spends in hospital
is a *discrete nonnegative* random variable.
As such, least mean squares might not be an ideal approach either.
This sort of time-to-event modeling
comes with a host of other complications that are dealt with
in a specialized subfield called *survival modeling*.

The point here is not to overwhelm you but just
to let you know that there is a lot more to estimation
than simply minimizing squared errors.
And more broadly, there is a lot more to supervised learning than regression.
In this section, we focus on *classification* problems
where we put aside *how much?* questions
and instead focus on *which category?* questions.



* Does this email belong in the spam folder or the inbox?
* Is this customer more likely to sign up
  or not to sign up for a subscription service?
* Does this image depict a donkey, a dog, a cat, or a rooster?
* Which movie is Aston most likely to watch next?
* Which section of the book are you going to read next?

Colloquially, machine learning practitioners
overload the word *classification*
to describe two subtly different problems:
(i) those where we are interested only in
hard assignments of examples to categories (classes);
and (ii) those where we wish to make soft assignments,
i.e., to assess the probability that each category applies.
The distinction tends to get blurred, in part,
because often, even when we only care about hard assignments,
we still use models that make soft assignments.

Even more, there are cases where more than one label might be true.
For instance, a news article might simultaneously cover
the topics of entertainment, business, and space flight,
but not the topics of medicine or sports.
Thus, categorizing it into one of the above categories
on their own would not be very useful.
This problem is commonly known as [multi-label classification](https://en.wikipedia.org/wiki/Multi-label_classification).
See :citet:`Tsoumakas.Katakis.2007` for an overview
and :citet:`Huang.Xu.Yu.2015`
for an effective algorithm when tagging images.

## Classification
:label:`subsec_classification-problem`

To get our feet wet, let's start with
a simple image classification problem.
Here, each input consists of a $2\times2$ grayscale image.
We can represent each pixel value with a single scalar,
giving us four features $x_1, x_2, x_3, x_4$.
Further, let's assume that each image belongs to one
among the categories "cat", "chicken", and "dog".

Next, we have to choose how to represent the labels.
We have two obvious choices.
Perhaps the most natural impulse would be
to choose $y \in \{1, 2, 3\}$,
where the integers represent
$\{\textrm{dog}, \textrm{cat}, \textrm{chicken}\}$ respectively.
This is a great way of *storing* such information on a computer.
If the categories had some natural ordering among them,
say if we were trying to predict
$\{\textrm{baby}, \textrm{toddler}, \textrm{adolescent}, \textrm{young adult}, \textrm{adult}, \textrm{geriatric}\}$,
then it might even make sense to cast this as
an [ordinal regression](https://en.wikipedia.org/wiki/Ordinal_regression) problem
and keep the labels in this format.
See :citet:`Moon.Smola.Chang.ea.2010` for an overview
of different types of ranking loss functions
and :citet:`Beutel.Murray.Faloutsos.ea.2014` for a Bayesian approach
that addresses responses with more than one mode.

In general, classification problems do not come
with natural orderings among the classes.
Fortunately, statisticians long ago invented a simple way
to represent categorical data: the *one-hot encoding*.
A one-hot encoding is a vector
with as many components as we have categories.
The component corresponding to a particular instance's category is set to 1
and all other components are set to 0.
In our case, a label $y$ would be a three-dimensional vector,
with $(1, 0, 0)$ corresponding to "cat", $(0, 1, 0)$ to "chicken",
and $(0, 0, 1)$ to "dog":

$$y \in \{(1, 0, 0), (0, 1, 0), (0, 0, 1)\}.$$

### Linear Model

In order to estimate the conditional probabilities
associated with all the possible classes,
we need a model with multiple outputs, one per class.
To address classification with linear models,
we will need as many affine functions as we have outputs.
Strictly speaking, we only need one fewer,
since the final category has to be the difference
between $1$ and the sum of the other categories,
but for reasons of symmetry
we use a slightly redundant parametrization.
Each output corresponds to its own affine function.
In our case, since we have 4 features and 3 possible output categories,
we need 12 scalars to represent the weights ($w$ with subscripts),
and 3 scalars to represent the biases ($b$ with subscripts). This yields:

$$
\begin{aligned}
o_1 &= x_1 w_{11} + x_2 w_{12} + x_3 w_{13} + x_4 w_{14} + b_1,\\
o_2 &= x_1 w_{21} + x_2 w_{22} + x_3 w_{23} + x_4 w_{24} + b_2,\\
o_3 &= x_1 w_{31} + x_2 w_{32} + x_3 w_{33} + x_4 w_{34} + b_3.
\end{aligned}
$$

The corresponding neural network diagram
is shown in :numref:`fig_softmaxreg`.
Just as in linear regression,
we use a single-layer neural network.
And since the calculation of each output, $o_1, o_2$, and $o_3$,
depends on every input, $x_1$, $x_2$, $x_3$, and $x_4$,
the output layer can also be described as a *fully connected layer*.

![Softmax regression is a single-layer neural network.](../img/softmaxreg.svg)
:label:`fig_softmaxreg`

For a more concise notation we use vectors and matrices:
$\mathbf{o} = \mathbf{W} \mathbf{x} + \mathbf{b}$ is
much better suited for mathematics and code.
Note that we have gathered all of our weights into a $3 \times 4$ matrix and all biases
$\mathbf{b} \in \mathbb{R}^3$ in a vector.

### The Softmax
:label:`subsec_softmax_operation`

Assuming a suitable loss function,
we could try, directly, to minimize the difference
between $\mathbf{o}$ and the labels $\mathbf{y}$.
While it turns out that treating classification
as a vector-valued regression problem works surprisingly well,
it is nonetheless unsatisfactory in the following ways:

* There is no guarantee that the outputs $o_i$ sum up to $1$ in the way we expect probabilities to behave.
* There is no guarantee that the outputs $o_i$ are even nonnegative, even if their outputs sum up to $1$, or that they do not exceed $1$.

Both aspects render the estimation problem difficult to solve
and the solution very brittle to outliers.
For instance, if we assume that there
is a positive linear dependency
between the number of bedrooms and the likelihood
that someone will buy a house,
the probability might exceed $1$
when it comes to buying a mansion!
As such, we need a mechanism to "squish" the outputs.

There are many ways we might accomplish this goal.
For instance, we could assume that the outputs
$\mathbf{o}$ are corrupted versions of $\mathbf{y}$,
where the corruption occurs by means of adding noise $\boldsymbol{\epsilon}$
drawn from a normal distribution.
In other words, $\mathbf{y} = \mathbf{o} + \boldsymbol{\epsilon}$,
where $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$.
This is the so-called [probit model](https://en.wikipedia.org/wiki/Probit_model),
first introduced by :citet:`Fechner.1860`.
While appealing, it does not work quite as well
nor lead to a particularly nice optimization problem,
when compared to the softmax.

Another way to accomplish this goal
(and to ensure nonnegativity) is to use
an exponential function $P(y = i) \propto \exp o_i$.
This does indeed satisfy the requirement
that the conditional class probability
increases with increasing $o_i$, it is monotonic,
and all probabilities are nonnegative.
We can then transform these values so that they add up to $1$
by dividing each by their sum.
This process is called *normalization*.
Putting these two pieces together
gives us the *softmax* function:

$$\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{o}) \quad \textrm{where}\quad \hat{y}_i = \frac{\exp(o_i)}{\sum_j \exp(o_j)}.$$
:eqlabel:`eq_softmax_y_and_o`

Note that the largest coordinate of $\mathbf{o}$
corresponds to the most likely class according to $\hat{\mathbf{y}}$.
Moreover, because the softmax operation
preserves the ordering among its arguments,
we do not need to compute the softmax
to determine which class has been assigned the highest probability. Thus,

$$
\operatorname*{argmax}_j \hat y_j = \operatorname*{argmax}_j o_j.
$$


The idea of a softmax dates back to :citet:`Gibbs.1902`,
who adapted ideas from physics.
Dating even further back, Boltzmann,
the father of modern statistical physics,
used this trick to model a distribution
over energy states in gas molecules.
In particular, he discovered that the prevalence
of a state of energy in a thermodynamic ensemble,
such as the molecules in a gas,
is proportional to $\exp(-E/kT)$.
Here, $E$ is the energy of a state,
$T$ is the temperature, and $k$ is the Boltzmann constant.
When statisticians talk about increasing or decreasing
the "temperature" of a statistical system,
they refer to changing $T$
in order to favor lower or higher energy states.
Following Gibbs' idea, energy equates to error.
Energy-based models :cite:`Ranzato.Boureau.Chopra.ea.2007`
use this point of view when describing
problems in deep learning.

### Vectorization
:label:`subsec_softmax_vectorization`

To improve computational efficiency,
we vectorize calculations in minibatches of data.
Assume that we are given a minibatch $\mathbf{X} \in \mathbb{R}^{n \times d}$
of $n$ examples with dimensionality (number of inputs) $d$.
Moreover, assume that we have $q$ categories in the output.
Then the weights satisfy $\mathbf{W} \in \mathbb{R}^{d \times q}$
and the bias satisfies $\mathbf{b} \in \mathbb{R}^{1\times q}$.

$$ \begin{aligned} \mathbf{O} &= \mathbf{X} \mathbf{W} + \mathbf{b}, \\ \hat{\mathbf{Y}} & = \mathrm{softmax}(\mathbf{O}). \end{aligned} $$
:eqlabel:`eq_minibatch_softmax_reg`

This accelerates the dominant operation into
a matrix--matrix product $\mathbf{X} \mathbf{W}$.
Moreover, since each row in $\mathbf{X}$ represents a data example,
the softmax operation itself can be computed *rowwise*:
for each row of $\mathbf{O}$, exponentiate all entries
and then normalize them by the sum.
Note, though, that care must be taken
to avoid exponentiating and taking logarithms of large numbers,
since this can cause numerical overflow or underflow.
Deep learning frameworks take care of this automatically.

## Loss Function
:label:`subsec_softmax-regression-loss-func`

Now that we have a mapping from features $\mathbf{x}$
to probabilities $\mathbf{\hat{y}}$,
we need a way to optimize the accuracy of this mapping.
We will rely on maximum likelihood estimation,
the very same method that we encountered
when providing a probabilistic justification
for the mean squared error loss in
:numref:`subsec_normal_distribution_and_squared_loss`.

### Log-Likelihood

The softmax function gives us a vector $\hat{\mathbf{y}}$,
which we can interpret as the (estimated) conditional probabilities
of each class, given any input $\mathbf{x}$,
such as $\hat{y}_1$ = $P(y=\textrm{cat} \mid \mathbf{x})$.
In the following we assume that for a dataset
with features $\mathbf{X}$ the labels $\mathbf{Y}$
are represented using a one-hot encoding label vector.
We can compare the estimates with reality
by checking how probable the actual classes are
according to our model, given the features:

$$
P(\mathbf{Y} \mid \mathbf{X}) = \prod_{i=1}^n P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}).
$$

We are allowed to use the factorization
since we assume that each label is drawn independently
from its respective distribution $P(\mathbf{y}\mid\mathbf{x}^{(i)})$.
Since maximizing the product of terms is awkward,
we take the negative logarithm to obtain the equivalent problem
of minimizing the negative log-likelihood:

$$
-\log P(\mathbf{Y} \mid \mathbf{X}) = \sum_{i=1}^n -\log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})
= \sum_{i=1}^n l(\mathbf{y}^{(i)}, \hat{\mathbf{y}}^{(i)}),
$$

where for any pair of label $\mathbf{y}$
and model prediction $\hat{\mathbf{y}}$
over $q$ classes, the loss function $l$ is

$$ l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j. $$
:eqlabel:`eq_l_cross_entropy`

For reasons explained later on,
the loss function in :eqref:`eq_l_cross_entropy`
is commonly called the *cross-entropy loss*.
Since $\mathbf{y}$ is a one-hot vector of length $q$,
the sum over all its coordinates $j$ vanishes for all but one term.
Note that the loss $l(\mathbf{y}, \hat{\mathbf{y}})$
is bounded from below by $0$
whenever $\hat{\mathbf{y}}$ is a probability vector:
no single entry is larger than $1$,
hence their negative logarithm cannot be lower than $0$;
$l(\mathbf{y}, \hat{\mathbf{y}}) = 0$ only if we predict
the actual label with *certainty*.
This can never happen for any finite setting of the weights
because taking a softmax output towards $1$
requires taking the corresponding input $o_i$ to infinity
(or all other outputs $o_j$ for $j \neq i$ to negative infinity).
Even if our model could assign an output probability of $0$,
any error made when assigning such high confidence
would incur infinite loss ($-\log 0 = \infty$).


### Softmax and Cross-Entropy Loss
:label:`subsec_softmax_and_derivatives`

Since the softmax function
and the corresponding cross-entropy loss are so common,
it is worth understanding a bit better how they are computed.
Plugging :eqref:`eq_softmax_y_and_o` into the definition of the loss
in :eqref:`eq_l_cross_entropy`
and using the definition of the softmax we obtain

$$
\begin{aligned}
l(\mathbf{y}, \hat{\mathbf{y}}) &=  - \sum_{j=1}^q y_j \log \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} \\
&= \sum_{j=1}^q y_j \log \sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j \\
&= \log \sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j.
\end{aligned}
$$

To understand a bit better what is going on,
consider the derivative with respect to any logit $o_j$. We get

$$
\partial_{o_j} l(\mathbf{y}, \hat{\mathbf{y}}) = \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} - y_j = \mathrm{softmax}(\mathbf{o})_j - y_j.
$$

In other words, the derivative is the difference
between the probability assigned by our model,
as expressed by the softmax operation,
and what actually happened, as expressed
by elements in the one-hot label vector.
In this sense, it is very similar
to what we saw in regression,
where the gradient was the difference
between the observation $y$ and estimate $\hat{y}$.
This is not a coincidence.
In any exponential family model,
the gradients of the log-likelihood are given by precisely this term.
This fact makes computing gradients easy in practice.

Now consider the case where we observe not just a single outcome
but an entire distribution over outcomes.
We can use the same representation as before for the label $\mathbf{y}$.
The only difference is that rather
than a vector containing only binary entries,
say $(0, 0, 1)$, we now have a generic probability vector,
say $(0.1, 0.2, 0.7)$.
The math that we used previously to define the loss $l$
in :eqref:`eq_l_cross_entropy`
still works well,
just that the interpretation is slightly more general.
It is the expected value of the loss for a distribution over labels.
This loss is called the *cross-entropy loss* and it is
one of the most commonly used losses for classification problems.
We can demystify the name by introducing just the basics of information theory.
In a nutshell, it measures the number of bits needed to encode what we see, $\mathbf{y}$,
relative to what we predict that should happen, $\hat{\mathbf{y}}$.
We provide a very basic explanation in the following. For further
details on information theory see
:citet:`Cover.Thomas.1999` or :citet:`mackay2003information`.



## Information Theory Basics
:label:`subsec_info_theory_basics`

Many deep learning papers use intuition and terms from information theory.
To make sense of them, we need some common language.
This is a survival guide.
*Information theory* deals with the problem
of encoding, decoding, transmitting,
and manipulating information (also known as data).

### Entropy

The central idea in information theory is to quantify the
amount of information contained in data.
This places a  limit on our ability to compress data.
For a distribution $P$ its *entropy*, $H[P]$, is defined as:

$$H[P] = \sum_j - P(j) \log P(j).$$
:eqlabel:`eq_softmax_reg_entropy`

One of the fundamental theorems of information theory states
that in order to encode data drawn randomly from the distribution $P$,
we need at least $H[P]$ "nats" to encode it :cite:`Shannon.1948`.
If you wonder what a "nat" is, it is the equivalent of bit
but when using a code with base $e$ rather than one with base 2.
Thus, one nat is $\frac{1}{\log(2)} \approx 1.44$ bit.


### Surprisal

You might be wondering what compression has to do with prediction.
Imagine that we have a stream of data that we want to compress.
If it is always easy for us to predict the next token,
then this data is easy to compress.
Take the extreme example where every token in the stream
always takes the same value.
That is a very boring data stream!
And not only it is boring, but it is also easy to predict.
Because the tokens are always the same,
we do not have to transmit any information
to communicate the contents of the stream.
Easy to predict, easy to compress.

However if we cannot perfectly predict every event,
then we might sometimes be surprised.
Our surprise is greater when an event is assigned lower probability.
Claude Shannon settled on $\log \frac{1}{P(j)} = -\log P(j)$
to quantify one's *surprisal* at observing an event $j$
having assigned it a (subjective) probability $P(j)$.
The entropy defined in :eqref:`eq_softmax_reg_entropy`
is then the *expected surprisal*
when one assigned the correct probabilities
that truly match the data-generating process.


### Cross-Entropy Revisited

So if entropy is the level of surprise experienced
by someone who knows the true probability,
then you might be wondering, what is cross-entropy?
The cross-entropy *from* $P$ *to* $Q$, denoted $H(P, Q)$,
is the expected surprisal of an observer with subjective probabilities $Q$
upon seeing data that was actually generated according to probabilities $P$.
This is given by $H(P, Q) \stackrel{\textrm{def}}{=} \sum_j - P(j) \log Q(j)$.
The lowest possible cross-entropy is achieved when $P=Q$.
In this case, the cross-entropy from $P$ to $Q$ is $H(P, P)= H(P)$.

In short, we can think of the cross-entropy classification objective
in two ways: (i) as maximizing the likelihood of the observed data;
and (ii) as minimizing our surprisal (and thus the number of bits)
required to communicate the labels.

## Summary and Discussion

In this section, we encountered the first nontrivial loss function,
allowing us to optimize over *discrete* output spaces.
Key in its design was that we took a probabilistic approach,
treating discrete categories as instances of draws from a probability distribution.
As a side effect, we encountered the softmax,
a convenient activation function that transforms
outputs of an ordinary neural network layer
into valid discrete probability distributions.
We saw that the derivative of the cross-entropy loss
when combined with softmax
behaves very similarly
to the derivative of squared error;
namely by taking the difference between
the expected behavior and its prediction.
And, while we were only able to
scratch the very surface of it,
we encountered exciting connections
to statistical physics and information theory.

While this is enough to get you on your way,
and hopefully enough to whet your appetite,
we hardly dived deep here.
Among other things, we skipped over computational considerations.
Specifically, for any fully connected layer with $d$ inputs and $q$ outputs,
the parametrization and computational cost is $\mathcal{O}(dq)$,
which can be prohibitively high in practice.
Fortunately, this cost of transforming $d$ inputs into $q$ outputs
can be reduced through approximation and compression.
For instance Deep Fried Convnets :cite:`Yang.Moczulski.Denil.ea.2015`
uses a combination of permutations,
Fourier transforms, and scaling
to reduce the cost from quadratic to log-linear.
Similar techniques work for more advanced
structural matrix approximations :cite:`sindhwani2015structured`.
Lastly, we can use quaternion-like decompositions
to reduce the cost to $\mathcal{O}(\frac{dq}{n})$,
again if we are willing to trade off a small amount of accuracy
for computational and storage cost :cite:`Zhang.Tay.Zhang.ea.2021`
based on a compression factor $n$.
This is an active area of research.
What makes it challenging is that
we do not necessarily strive
for the most compact representation
or the smallest number of floating point operations
but rather for the solution
that can be executed most efficiently on modern GPUs.

## Exercises

1. We can explore the connection between exponential families and softmax in some more depth.
    1. Compute the second derivative of the cross-entropy loss $l(\mathbf{y},\hat{\mathbf{y}})$ for softmax.
    1. Compute the variance of the distribution given by $\mathrm{softmax}(\mathbf{o})$ and show that it matches the second derivative computed above.
1. Assume that we have three classes which occur with equal probability, i.e., the probability vector is $(\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$.
    1. What is the problem if we try to design a binary code for it?
    1. Can you design a better code? Hint: what happens if we try to encode two independent observations? What if we encode $n$ observations jointly?
1. When encoding signals transmitted over a physical wire, engineers do not always use binary codes. For instance, [PAM-3](https://en.wikipedia.org/wiki/Ternary_signal) uses three signal levels $\{-1, 0, 1\}$ as opposed to two levels $\{0, 1\}$. How many ternary units do you need to transmit an integer in the range $\{0, \ldots, 7\}$? Why might this be a better idea in terms of electronics?
1. The [Bradley--Terry model](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model) uses
a logistic model to capture preferences. For a user to choose between apples and oranges one
assumes scores $o_{\textrm{apple}}$ and $o_{\textrm{orange}}$. Our requirements are that larger scores should lead to a higher likelihood in choosing the associated item and that
the item with the largest score is the most likely one to be chosen :cite:`Bradley.Terry.1952`.
    1. Prove that softmax satisfies this requirement.
    1. What happens if you want to allow for a default option of choosing neither apples nor oranges? Hint: now the user has three choices.
1. Softmax gets its name from the following mapping: $\textrm{RealSoftMax}(a, b) = \log (\exp(a) + \exp(b))$.
    1. Prove that $\textrm{RealSoftMax}(a, b) > \mathrm{max}(a, b)$.
    1. How small can you make the difference between both functions? Hint: without loss of
    generality you can set $b = 0$ and $a \geq b$.
    1. Prove that this holds for $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b)$, provided that $\lambda > 0$.
    1. Show that for $\lambda \to \infty$ we have $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b) \to \mathrm{max}(a, b)$.
    1. Construct an analogous softmin function.
    1. Extend this to more than two numbers.
1. The function $g(\mathbf{x}) \stackrel{\textrm{def}}{=} \log \sum_i \exp x_i$ is sometimes also referred to as the [log-partition function](https://en.wikipedia.org/wiki/Partition_function_(mathematics)).
    1. Prove that the function is convex. Hint: to do so, use the fact that the first derivative amounts to the probabilities from the softmax function and show that the second derivative is the variance.
    1. Show that $g$ is translation invariant, i.e., $g(\mathbf{x} + b) = g(\mathbf{x})$.
    1. What happens if some of the coordinates $x_i$ are very large? What happens if they're all very small?
    1. Show that if we choose $b = \mathrm{max}_i x_i$ we end up with a numerically stable implementation.
1. Assume that we have some probability distribution $P$. Suppose we pick another distribution $Q$ with $Q(i) \propto P(i)^\alpha$ for $\alpha > 0$.
    1. Which choice of $\alpha$ corresponds to doubling the temperature? Which choice corresponds to halving it?
    1. What happens if we let the temperature approach $0$?
    1. What happens if we let the temperature approach $\infty$?

[Discussions](https://discuss.d2l.ai/t/46)



1. We can explore the connection between exponential families and softmax in some more depth.
    1. Compute the second derivative of the cross-entropy loss $l(\mathbf{y},\hat{\mathbf{y}})$ for softmax.
    1. Compute the variance of the distribution given by $\mathrm{softmax}(\mathbf{o})$ and show that it matches the second derivative computed above.


# Connection Between Exponential Families and Softmax

Let's explore the connection between exponential families and softmax by examining the second derivatives of the cross-entropy loss and relating them to the variance of the softmax distribution.

## 1. Second Derivative of Cross-Entropy Loss

First, let's recall the key equations:
- Cross-entropy loss: $l(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{j=1}^q y_j \log \hat{y}_j$
- Softmax function: $\hat{y}_i = \frac{\exp(o_i)}{\sum_k \exp(o_k)}$
- First derivative: $\frac{\partial l}{\partial o_i} = \hat{y}_i - y_i$

To find the second derivative, we need to differentiate the first derivative with respect to $o_j$:

$$\frac{\partial^2 l}{\partial o_j \partial o_i} = \frac{\partial}{\partial o_j}(\hat{y}_i - y_i)$$

Since $y_i$ is a constant (the true label), its derivative is zero. So we need to focus on $\frac{\partial \hat{y}_i}{\partial o_j}$.

Using the quotient rule for the softmax function:

$$\frac{\partial \hat{y}_i}{\partial o_j} = \frac{\partial}{\partial o_j}\left(\frac{\exp(o_i)}{\sum_k \exp(o_k)}\right)$$

Two cases arise:

1. When $i = j$:
   $$\frac{\partial \hat{y}_i}{\partial o_i} = \hat{y}_i(1 - \hat{y}_i)$$

2. When $i \neq j$:
   $$\frac{\partial \hat{y}_i}{\partial o_j} = -\hat{y}_i\hat{y}_j$$

Therefore, the second derivative of the cross-entropy loss is:

$$\frac{\partial^2 l}{\partial o_j \partial o_i} = 
\begin{cases}
\hat{y}_i(1 - \hat{y}_i) & \text{if } i = j \\
-\hat{y}_i\hat{y}_j & \text{if } i \neq j
\end{cases}$$

We can write this more compactly using the Kronecker delta $\delta_{ij}$:

$$\frac{\partial^2 l}{\partial o_j \partial o_i} = \hat{y}_i(\delta_{ij} - \hat{y}_j)$$

## 2. Variance of the Softmax Distribution

Now, let's compute the variance of the softmax distribution. The softmax function gives us probabilities $\hat{y}_i$, and we can interpret this as a categorical distribution.

For a categorical distribution, the covariance matrix elements are:

$$\text{Cov}(I_i, I_j) = 
\begin{cases}
\hat{y}_i(1 - \hat{y}_i) & \text{if } i = j \\
-\hat{y}_i\hat{y}_j & \text{if } i \neq j
\end{cases}$$

where $I_i$ is the indicator variable that equals 1 if category $i$ is selected and 0 otherwise.

This gives us the same expression as our second derivative:

$$\text{Cov}(I_i, I_j) = \hat{y}_i(\delta_{ij} - \hat{y}_j)$$

Therefore, the second derivative of the cross-entropy loss with respect to the logits is exactly equal to the covariance matrix of the categorical distribution defined by the softmax probabilities.

## Intuitive Explanation

This connection reveals a profound relationship between the geometry of the loss function and the statistical properties of the model:

1. The curvature of the loss function (represented by the second derivatives) directly corresponds to the uncertainty in our predictions (represented by the variance).

2. When the model is very certain (probabilities close to 0 or 1), the variance is small, and the loss function is relatively flat in those regions.

3. Conversely, when the model is uncertain (probabilities closer to uniform), the variance is high, and the loss function has high curvature.

This relationship is a special case of a more general connection between exponential family distributions and their corresponding loss functions in maximum likelihood estimation.

Let me break down both parts more thoroughly for your understanding:

## Part 1: Second Derivative of Cross-Entropy Loss (Step by Step)

Let's start with the cross-entropy loss:
$$l(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{j=1}^q y_j \log \hat{y}_j$$

Where $\hat{y}_i$ is the softmax function:
$$\hat{y}_i = \frac{\exp(o_i)}{\sum_k \exp(o_k)}$$

### Step 1: Find the first derivative
We first need to find $\frac{\partial l}{\partial o_i}$:

$$\frac{\partial l}{\partial o_i} = \frac{\partial}{\partial o_i} \left(-\sum_{j=1}^q y_j \log \hat{y}_j \right)$$

Since the only terms that depend on $o_i$ are those involving $\hat{y}_j$ (because changing $o_i$ affects all softmax outputs due to the denominator), we need to use the chain rule:

$$\frac{\partial l}{\partial o_i} = -\sum_{j=1}^q y_j \frac{\partial \log \hat{y}_j}{\partial \hat{y}_j} \frac{\partial \hat{y}_j}{\partial o_i}$$

$$\frac{\partial l}{\partial o_i} = -\sum_{j=1}^q y_j \frac{1}{\hat{y}_j} \frac{\partial \hat{y}_j}{\partial o_i}$$

Now we need to find $\frac{\partial \hat{y}_j}{\partial o_i}$ for each $j$:

For the case when $j = i$:
$$\frac{\partial \hat{y}_i}{\partial o_i} = \frac{\partial}{\partial o_i}\left(\frac{\exp(o_i)}{\sum_k \exp(o_k)}\right)$$

Using the quotient rule:
$$\frac{\partial \hat{y}_i}{\partial o_i} = \frac{\exp(o_i) \cdot \sum_k \exp(o_k) - \exp(o_i) \cdot \exp(o_i)}{(\sum_k \exp(o_k))^2}$$

$$\frac{\partial \hat{y}_i}{\partial o_i} = \frac{\exp(o_i)}{\sum_k \exp(o_k)} \cdot \left(1 - \frac{\exp(o_i)}{\sum_k \exp(o_k)}\right)$$

$$\frac{\partial \hat{y}_i}{\partial o_i} = \hat{y}_i (1 - \hat{y}_i)$$

For the case when $j \neq i$:
$$\frac{\partial \hat{y}_j}{\partial o_i} = \frac{\partial}{\partial o_i}\left(\frac{\exp(o_j)}{\sum_k \exp(o_k)}\right)$$

The numerator doesn't depend on $o_i$ (when $j \neq i$), so:
$$\frac{\partial \hat{y}_j}{\partial o_i} = \exp(o_j) \cdot \frac{\partial}{\partial o_i}\left(\frac{1}{\sum_k \exp(o_k)}\right)$$

$$\frac{\partial \hat{y}_j}{\partial o_i} = \exp(o_j) \cdot \frac{-\exp(o_i)}{(\sum_k \exp(o_k))^2}$$

$$\frac{\partial \hat{y}_j}{\partial o_i} = -\frac{\exp(o_j)}{\sum_k \exp(o_k)} \cdot \frac{\exp(o_i)}{\sum_k \exp(o_k)}$$

$$\frac{\partial \hat{y}_j}{\partial o_i} = -\hat{y}_j \hat{y}_i$$

Now back to the first derivative:
$$\frac{\partial l}{\partial o_i} = -\sum_{j=1}^q y_j \frac{1}{\hat{y}_j} \frac{\partial \hat{y}_j}{\partial o_i}$$

Substituting what we found:
$$\frac{\partial l}{\partial o_i} = -y_i \frac{1}{\hat{y}_i} \hat{y}_i (1 - \hat{y}_i) - \sum_{j \neq i} y_j \frac{1}{\hat{y}_j} (-\hat{y}_j \hat{y}_i)$$

$$\frac{\partial l}{\partial o_i} = -y_i (1 - \hat{y}_i) + \sum_{j \neq i} y_j \hat{y}_i$$

$$\frac{\partial l}{\partial o_i} = -y_i + y_i\hat{y}_i + \hat{y}_i\sum_{j \neq i} y_j$$

Since $\sum_{j=1}^q y_j = 1$ (because $\mathbf{y}$ is a one-hot encoded vector), we have $\sum_{j \neq i} y_j = 1 - y_i$:

$$\frac{\partial l}{\partial o_i} = -y_i + y_i\hat{y}_i + \hat{y}_i(1 - y_i)$$

$$\frac{\partial l}{\partial o_i} = -y_i + y_i\hat{y}_i + \hat{y}_i - y_i\hat{y}_i$$

$$\frac{\partial l}{\partial o_i} = \hat{y}_i - y_i$$

### Step 2: Find the second derivative
Now we compute $\frac{\partial^2 l}{\partial o_j \partial o_i}$ by differentiating the first derivative with respect to $o_j$:

$$\frac{\partial^2 l}{\partial o_j \partial o_i} = \frac{\partial}{\partial o_j}(\hat{y}_i - y_i)$$

Since $y_i$ is constant, this reduces to:
$$\frac{\partial^2 l}{\partial o_j \partial o_i} = \frac{\partial \hat{y}_i}{\partial o_j}$$

We've already computed these derivatives above. We have two cases:

When $i = j$:
$$\frac{\partial^2 l}{\partial o_i \partial o_i} = \frac{\partial \hat{y}_i}{\partial o_i} = \hat{y}_i (1 - \hat{y}_i)$$

When $i \neq j$:
$$\frac{\partial^2 l}{\partial o_j \partial o_i} = \frac{\partial \hat{y}_i}{\partial o_j} = -\hat{y}_i \hat{y}_j$$

So the general formula for the second derivative is:
$$\frac{\partial^2 l}{\partial o_j \partial o_i} = 
\begin{cases}
\hat{y}_i(1 - \hat{y}_i) & \text{if } i = j \\
-\hat{y}_i\hat{y}_j & \text{if } i \neq j
\end{cases}$$

Using the Kronecker delta notation ($\delta_{ij} = 1$ if $i = j$ and 0 otherwise), we can express this more compactly:
$$\frac{\partial^2 l}{\partial o_j \partial o_i} = \hat{y}_i(\delta_{ij} - \hat{y}_j)$$

## Part 2: Variance of the Softmax Distribution (Step by Step)

Now let's compute the variance of the categorical distribution defined by the softmax probabilities.

A categorical distribution with $q$ classes and probabilities $\hat{y}_1, \hat{y}_2, \ldots, \hat{y}_q$ can be represented using indicator random variables $I_1, I_2, \ldots, I_q$, where $I_i = 1$ if category $i$ is chosen and 0 otherwise.

### Step 1: Calculate the mean of each indicator
The mean (expected value) of each indicator variable is:
$$E[I_i] = 1 \cdot \hat{y}_i + 0 \cdot (1 - \hat{y}_i) = \hat{y}_i$$

### Step 2: Calculate the variance and covariance
For categorical distributions, we need to find the covariance matrix. The elements of this matrix are:

For the variance (when $i = j$):
$$\text{Var}(I_i) = E[I_i^2] - E[I_i]^2$$

Since $I_i$ is either 0 or 1, we have $I_i^2 = I_i$, so:
$$\text{Var}(I_i) = E[I_i] - E[I_i]^2 = \hat{y}_i - \hat{y}_i^2 = \hat{y}_i(1 - \hat{y}_i)$$

For the covariance (when $i \neq j$):
$$\text{Cov}(I_i, I_j) = E[I_i I_j] - E[I_i]E[I_j]$$

Since we can only select one category at a time, we know that $I_i$ and $I_j$ cannot both be 1 simultaneously. Therefore, $I_i I_j = 0$ always when $i \neq j$. So:
$$\text{Cov}(I_i, I_j) = 0 - \hat{y}_i\hat{y}_j = -\hat{y}_i\hat{y}_j$$

Therefore, the elements of the covariance matrix are:
$$\text{Cov}(I_i, I_j) = 
\begin{cases}
\hat{y}_i(1 - \hat{y}_i) & \text{if } i = j \\
-\hat{y}_i\hat{y}_j & \text{if } i \neq j
\end{cases}$$

Or using the Kronecker delta notation:
$$\text{Cov}(I_i, I_j) = \hat{y}_i(\delta_{ij} - \hat{y}_j)$$

This matches exactly with the second derivative of the cross-entropy loss that we derived earlier:
$$\frac{\partial^2 l}{\partial o_j \partial o_i} = \hat{y}_i(\delta_{ij} - \hat{y}_j) = \text{Cov}(I_i, I_j)$$

This confirms the deep connection between the geometry of the loss function (its curvature) and the statistical properties of the model (the variance of its predictions).

2. Assume that we have three classes which occur with equal probability, i.e., the probability vector is $(\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$.
    1. What is the problem if we try to design a binary code for it?
    1. Can you design a better code? Hint: what happens if we try to encode two independent observations? What if we encode $n$ observations jointly?


# Binary Coding for a Three-Class Distribution

## 1. Problems with Binary Coding for Three Equally Likely Classes

When we have three classes with equal probabilities $(\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$, designing an optimal binary code becomes problematic for several reasons:

### Theoretical Issue: Information Theory Mismatch

From information theory, the entropy of this distribution is:
$$H = -\sum_{i=1}^{3} p_i \log_2 p_i = -3 \cdot \frac{1}{3} \log_2 \frac{1}{3} = \log_2 3 \approx 1.585 \text{ bits}$$

This means we need about 1.585 bits per symbol on average to encode this information optimally. However, binary codes assign whole numbers of bits to each symbol, forcing us to use at least 2 bits per symbol:
- Class 1: 00
- Class 2: 01
- Class 3: 10

The code "11" is unused, making this coding inefficient. Our coding efficiency is only $\frac{1.585}{2} \approx 79.25\%$, wasting approximately 20.75% of the transmitted bits.

### Practical Issue: Code Length Limitation

With a fixed-length binary code, we must use 2 bits per symbol, which exceeds the theoretical minimum of 1.585 bits. There's no way to achieve the theoretical minimum with a simple binary code for individual symbols because we're restricted to using whole numbers of bits.

## 2. Designing Better Codes

### Encoding Multiple Observations Together

The key insight is to encode multiple observations jointly rather than individually. This allows us to approach the theoretical limit of $\log_2 3 \approx 1.585$ bits per symbol.

#### Two Observations Case

Let's encode pairs of observations. With two observations, we have $3^2 = 9$ possible combinations, which requires $\lceil \log_2 9 \rceil = 4$ bits.

Encoding:
- (Class 1, Class 1): 0000
- (Class 1, Class 2): 0001
- (Class 1, Class 3): 0010
- (Class 2, Class 1): 0011
- (Class 2, Class 2): 0100
- (Class 2, Class 3): 0101
- (Class 3, Class 1): 0110
- (Class 3, Class 2): 0111
- (Class 3, Class 3): 1000

Now we're using 4 bits to encode 2 symbols, averaging 2 bits per symbol, which is still not optimal.

#### Three Observations Case

With three observations, we have $3^3 = 27$ possible combinations, requiring $\lceil \log_2 27 \rceil = 5$ bits.

Now we're using 5 bits to encode 3 symbols, averaging $\frac{5}{3} \approx 1.67$ bits per symbol, which is closer to the optimal 1.585 bits.

#### n Observations Case

As we encode more observations together, the efficiency improves:

For $n$ observations, we have $3^n$ combinations, requiring $\lceil \log_2 3^n \rceil = \lceil n \log_2 3 \rceil$ bits.

The average number of bits per symbol approaches the theoretical limit:
$$\lim_{n \to \infty} \frac{\lceil n \log_2 3 \rceil}{n} = \log_2 3 \approx 1.585 \text{ bits}$$

### Practical Examples

For concrete efficiency:
- $n = 5$: $\lceil 5 \log_2 3 \rceil = \lceil 7.925 \rceil = 8$ bits → $\frac{8}{5} = 1.6$ bits/symbol
- $n = 10$: $\lceil 10 \log_2 3 \rceil = \lceil 15.85 \rceil = 16$ bits → $\frac{16}{10} = 1.6$ bits/symbol
- $n = 100$: $\lceil 100 \log_2 3 \rceil = \lceil 158.5 \rceil = 159$ bits → $\frac{159}{100} = 1.59$ bits/symbol

The larger the block size ($n$) we encode together, the closer we get to the theoretical limit of 1.585 bits per symbol, achieving better compression efficiency.

This approach, known as block coding, illustrates how grouping symbols together can overcome the limitations of simple binary coding for non-binary alphabets, a core principle in data compression algorithms.

For more explanation read colah's blog on information theory

3. When encoding signals transmitted over a physical wire, engineers do not always use binary codes. For instance, [PAM-3](https://en.wikipedia.org/wiki/Ternary_signal) uses three signal levels $\{-1, 0, 1\}$ as opposed to two levels $\{0, 1\}$. How many ternary units do you need to transmit an integer in the range $\{0, \ldots, 7\}$? Why might this be a better idea in terms of electronics?


# Ternary Signal Encoding for Data Transmission

## Calculating Ternary Units Required

To determine how many ternary units (trits) we need to encode integers in the range $\{0, \ldots, 7\}$, we need to find the smallest number $n$ such that $3^n \geq 8$.

Since we have 8 distinct values to represent:
- $3^1 = 3$ (insufficient)
- $3^2 = 9$ (sufficient)

Therefore, we need 2 ternary units (trits) to represent integers from 0 to 7.

The encoding could work as follows:
- $0 = (-1, -1)$ or $(0, 0)$ depending on the encoding scheme
- $1 = (-1, 0)$ or $(0, 1)$
- $2 = (-1, 1)$ or $(0, 2)$
- $3 = (0, -1)$ or $(1, 0)$
- $4 = (0, 0)$ or $(1, 1)$
- $5 = (0, 1)$ or $(1, 2)$
- $6 = (1, -1)$ or $(2, 0)$
- $7 = (1, 0)$ or $(2, 1)$

Note that we would only use 8 of the 9 possible combinations with a standard encoding.

## Advantages in Electronic Systems

There are several electronic advantages to using ternary signaling like PAM-3 instead of binary:

### 1. Bandwidth Efficiency

PAM-3 encodes $\log_2(3) \approx 1.585$ bits per symbol, which means it can transmit more information per unit time compared to binary signaling in the same bandwidth. For our example, 2 ternary symbols encode 8 values, while we would need 3 binary symbols for the same range.

### 2. Signal-to-Noise Ratio Considerations

While binary requires distinguishing between only two levels, ternary requires distinguishing between three levels, which demands better signal-to-noise ratio. However, this trade-off can be worthwhile in certain contexts.

### 3. DC Balance and Spectral Properties

The inclusion of the zero level in PAM-3 $\{-1, 0, 1\}$ allows for better DC balance in transmission lines. DC-balanced signals have little or no net DC component, which is important for:

- Allowing AC coupling (using capacitors to connect stages, blocking DC components)
- Reducing electromagnetic interference (EMI)
- Enabling simpler clock recovery circuits
- Preventing baseline wander in receivers

### 4. Reduced Power Consumption

In many electronic implementations, transitioning between adjacent signal levels (e.g., from -1 to 0 or from 0 to 1) consumes less power than making full swings (from -1 to 1). The intermediate zero level allows more transitions to be shorter, potentially reducing overall power consumption.

### 5. Reduced Electromagnetic Interference

Smaller voltage swings and more gradual transitions between levels can reduce high-frequency components in the signal spectrum, potentially reducing electromagnetic interference.

### 6. Real-World Applications

This approach is used in various modern interfaces:
- USB 3.0 SuperSpeed uses a form of ternary signaling
- Ethernet 1000BASE-T uses PAM-5 (five levels)
- Some memory interfaces use multi-level signaling

The trade-off between spectral efficiency, power consumption, and error resistance makes ternary signaling an excellent engineering choice for many physical communication channels, despite the increased complexity of the encoding and decoding circuitry.

4. The [Bradley--Terry model](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model) uses
a logistic model to capture preferences. For a user to choose between apples and oranges one
assumes scores $o_{\textrm{apple}}$ and $o_{\textrm{orange}}$. Our requirements are that larger scores should lead to a higher likelihood in choosing the associated item and that
the item with the largest score is the most likely one to be chosen :cite:`Bradley.Terry.1952`.
    1. Prove that softmax satisfies this requirement.
    1. What happens if you want to allow for a default option of choosing neither apples nor oranges? Hint: now the user has three choices.


# Bradley-Terry Model and Softmax Analysis

## 1. Proving Softmax Satisfies the Bradley-Terry Requirements

The Bradley-Terry model assumes that items have latent scores that determine the probability of selection. Let's prove that softmax satisfies the two key requirements:

1. **Higher scores lead to higher likelihood of selection**
2. **The item with the highest score is the most likely to be chosen**

Consider two items with scores $o_1$ and $o_2$. Using the softmax function, the probability of choosing item 1 is:

$$P(\text{choose item 1}) = \frac{\exp(o_1)}{\exp(o_1) + \exp(o_2)} = \frac{1}{1 + \exp(o_2 - o_1)}$$

And similarly for item 2:

$$P(\text{choose item 2}) = \frac{\exp(o_2)}{\exp(o_1) + \exp(o_2)} = \frac{1}{1 + \exp(o_1 - o_2)}$$

### Proof for Requirement 1:

To show that higher scores lead to higher likelihood, let's examine the derivative of the probability with respect to the score:

$$\frac{\partial P(\text{choose item 1})}{\partial o_1} = \frac{\exp(o_1) \cdot \exp(o_2)}{(\exp(o_1) + \exp(o_2))^2} > 0$$

Since this derivative is always positive, increasing $o_1$ always increases the probability of choosing item 1. Similarly, decreasing $o_1$ decreases this probability.

### Proof for Requirement 2:

To show that the item with the highest score is most likely to be chosen, let's compare the probabilities directly:

$$P(\text{choose item 1}) > P(\text{choose item 2})$$

$$\frac{\exp(o_1)}{\exp(o_1) + \exp(o_2)} > \frac{\exp(o_2)}{\exp(o_1) + \exp(o_2)}$$

This simplifies to:

$$\exp(o_1) > \exp(o_2)$$

Since the exponential function is strictly increasing, this is equivalent to:

$$o_1 > o_2$$

Therefore, the item with the highest score will always have the highest probability of being chosen, satisfying the second requirement.

## 2. Adding a Default "Neither" Option

If we want to allow for a default option of choosing neither apples nor oranges, we now have three choices with scores:
- $o_{\text{apple}}$
- $o_{\text{orange}}$
- $o_{\text{neither}}$

The softmax function naturally extends to accommodate this third option:

$$P(\text{choose apple}) = \frac{\exp(o_{\text{apple}})}{\exp(o_{\text{apple}}) + \exp(o_{\text{orange}}) + \exp(o_{\text{neither}})}$$

$$P(\text{choose orange}) = \frac{\exp(o_{\text{orange}})}{\exp(o_{\text{apple}}) + \exp(o_{\text{orange}}) + \exp(o_{\text{neither}})}$$

$$P(\text{choose neither}) = \frac{\exp(o_{\text{neither}})}{\exp(o_{\text{apple}}) + \exp(o_{\text{orange}}) + \exp(o_{\text{neither}})}$$

The "neither" option can be modeled in several ways:

1. **As a competitive alternative**: In this case, $o_{\text{neither}}$ is a score like any other, and could potentially be the highest.

2. **As a default/baseline option**: We could set $o_{\text{neither}} = 0$ (or any constant), making it a reference point against which the other options are evaluated.

3. **As a threshold model**: We could define that the "neither" option is chosen when both apple and orange scores fall below some threshold $\tau$. In the softmax framework, this could be implemented by setting:
   $$o_{\text{neither}} = \tau - \max(o_{\text{apple}}, o_{\text{orange}})$$

In all these approaches, the fundamental properties of softmax remain intact:
- Higher scores still lead to higher probabilities
- The option with the highest score is still the most likely to be chosen

A key insight from this extension is that the "neither" option doesn't require any special treatment in the softmax framework—it's simply treated as another option with its own score, demonstrating the flexibility of the softmax function for modeling choice preferences with any number of alternatives.

5. Softmax gets its name from the following mapping: $\textrm{RealSoftMax}(a, b) = \log (\exp(a) + \exp(b))$.
    1. Prove that $\textrm{RealSoftMax}(a, b) > \mathrm{max}(a, b)$.
    1. How small can you make the difference between both functions? Hint: without loss of
    generality you can set $b = 0$ and $a \geq b$.
    1. Prove that this holds for $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b)$, provided that $\lambda > 0$.
    1. Show that for $\lambda \to \infty$ we have $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b) \to \mathrm{max}(a, b)$.
    1. Construct an analogous softmin function.
    1. Extend this to more than two numbers.


we can prove it like this 

e^a + e^b >= e^(max(a,b))

Then assuming either a is greater than b and then other way around we can show e^a > 0 Which is true. Unless a tends to negative infinity.

# Understanding the RealSoftMax Function Properties

## 1. Proving that RealSoftMax(a, b) > max(a, b)

Let's prove that $\textrm{RealSoftMax}(a, b) = \log(\exp(a) + \exp(b)) > \max(a, b)$.

Without loss of generality, assume $a \geq b$ (if not, we can simply swap $a$ and $b$).
Then $\max(a, b) = a$.

We need to show: $\log(\exp(a) + \exp(b)) > a$

Taking $\exp$ of both sides (which preserves inequality since $\exp$ is strictly increasing):
$\exp(a) + \exp(b) > \exp(a)$

This simplifies to:
$\exp(b) > 0$

Since $\exp(b) > 0$ for any finite value of $b$, the inequality holds.

Therefore, $\textrm{RealSoftMax}(a, b) > \max(a, b)$ for all finite $a$ and $b$.

## 2. Minimizing the Difference Between RealSoftMax and max

We want to find how small the difference $\textrm{RealSoftMax}(a, b) - \max(a, b)$ can be.

As suggested, we can set $b = 0$ and assume $a \geq 0$. Then:
$\textrm{RealSoftMax}(a, 0) - \max(a, 0) = \log(\exp(a) + 1) - a$

For $a \leq 0$: $\log(\exp(a) + 1) - 0 = \log(\exp(a) + 1) > 0$, and this approaches $\log(1) = 0$ as $a \to -\infty$

For $a > 0$: $\log(\exp(a) + 1) - a = \log(1 + \exp(-a))$
As $a \to \infty$, this approaches $\log(1) = 0$

Therefore, the difference can be made arbitrarily small by taking $a$ very large (when $a > 0$) or very negative (when $a \leq 0$).

## 3. Proving the Property Holds for Scaled Inputs

We need to show that $\lambda^{-1}\textrm{RealSoftMax}(\lambda a, \lambda b) > \max(a, b)$ for $\lambda > 0$.

$\lambda^{-1}\textrm{RealSoftMax}(\lambda a, \lambda b) = \lambda^{-1}\log(\exp(\lambda a) + \exp(\lambda b))$

$= \lambda^{-1}\log(\exp(\lambda a)(1 + \exp(\lambda(b-a))))$ (assuming $a \geq b$ without loss of generality)

$= \lambda^{-1}(\lambda a + \log(1 + \exp(\lambda(b-a))))$

$= a + \lambda^{-1}\log(1 + \exp(\lambda(b-a)))$

Since $\log(1 + \exp(\lambda(b-a))) > 0$ for finite values, we have:
$a + \lambda^{-1}\log(1 + \exp(\lambda(b-a))) > a = \max(a, b)$

Therefore, the inequality holds for any $\lambda > 0$.

## 4. Limit as λ Approaches Infinity

We need to show that $\lim_{\lambda \to \infty} \lambda^{-1}\textrm{RealSoftMax}(\lambda a, \lambda b) = \max(a, b)$.

From our previous step:
$\lambda^{-1}\textrm{RealSoftMax}(\lambda a, \lambda b) = a + \lambda^{-1}\log(1 + \exp(\lambda(b-a)))$

Since $a \geq b$, we have $b-a \leq 0$, so $\exp(\lambda(b-a)) \to 0$ as $\lambda \to \infty$.

Thus, $\lambda^{-1}\log(1 + \exp(\lambda(b-a))) \to \lambda^{-1}\log(1) = 0$ as $\lambda \to \infty$.

Therefore:
$\lim_{\lambda \to \infty} \lambda^{-1}\textrm{RealSoftMax}(\lambda a, \lambda b) = a = \max(a, b)$

## 5. Constructing an Analogous softmin Function

Following the pattern of RealSoftMax, we can define:
$\textrm{RealSoftMin}(a, b) = -\log(\exp(-a) + \exp(-b))$

This function has the property that $\textrm{RealSoftMin}(a, b) < \min(a, b)$, and as we increase a scaling parameter $\lambda$, the function $\lambda^{-1}\textrm{RealSoftMin}(\lambda a, \lambda b)$ approaches $\min(a, b)$.

## 6. Extension to Multiple Numbers

The RealSoftMax function generalizes naturally to multiple inputs:
$\textrm{RealSoftMax}(a_1, a_2, \ldots, a_n) = \log(\sum_{i=1}^{n} \exp(a_i))$

Similarly, for the RealSoftMin:
$\textrm{RealSoftMin}(a_1, a_2, \ldots, a_n) = -\log(\sum_{i=1}^{n} \exp(-a_i))$

The same properties hold: 
- $\textrm{RealSoftMax}(a_1, a_2, \ldots, a_n) > \max(a_1, a_2, \ldots, a_n)$
- $\lim_{\lambda \to \infty} \lambda^{-1}\textrm{RealSoftMax}(\lambda a_1, \lambda a_2, \ldots, \lambda a_n) = \max(a_1, a_2, \ldots, a_n)$
- $\textrm{RealSoftMin}(a_1, a_2, \ldots, a_n) < \min(a_1, a_2, \ldots, a_n)$
- $\lim_{\lambda \to \infty} \lambda^{-1}\textrm{RealSoftMin}(\lambda a_1, \lambda a_2, \ldots, \lambda a_n) = \min(a_1, a_2, \ldots, a_n)$

These smooth approximations to the min and max functions are valuable in optimization problems where differentiability is required.

6. The function $g(\mathbf{x}) \stackrel{\textrm{def}}{=} \log \sum_i \exp x_i$ is sometimes also referred to as the [log-partition function](https://en.wikipedia.org/wiki/Partition_function_(mathematics)).
    1. Prove that the function is convex. Hint: to do so, use the fact that the first derivative amounts to the probabilities from the softmax function and show that the second derivative is the variance.
    1. Show that $g$ is translation invariant, i.e., $g(\mathbf{x} + b) = g(\mathbf{x})$.
    1. What happens if some of the coordinates $x_i$ are very large? What happens if they're all very small?
    1. Show that if we choose $b = \mathrm{max}_i x_i$ we end up with a numerically stable implementation.


# Properties of the Log-Partition Function

## 1. Proving Convexity of the Function

To prove that $g(\mathbf{x}) = \log \sum_i \exp x_i$ is convex, I'll analyze its derivatives.

First, let's compute the gradient (first derivative) of $g$:

$$\frac{\partial g(\mathbf{x})}{\partial x_j} = \frac{\exp(x_j)}{\sum_i \exp(x_i)}$$

This is precisely the softmax function, which gives us probability values (they're positive and sum to 1).

For the Hessian (second derivative), we have:

$$\frac{\partial^2 g(\mathbf{x})}{\partial x_k \partial x_j} = 
\begin{cases}
p_j(1-p_j) & \text{if } j = k \\
-p_j p_k & \text{if } j \neq k
\end{cases}$$

Where $p_j = \frac{\exp(x_j)}{\sum_i \exp(x_i)}$.

To prove convexity, I need to show that the Hessian is positive semidefinite. For any vector $\mathbf{v}$, we need:
$$\mathbf{v}^T H \mathbf{v} \geq 0$$

Computing this product:
$$\mathbf{v}^T H \mathbf{v} = \sum_j \sum_k v_j v_k \frac{\partial^2 g(\mathbf{x})}{\partial x_k \partial x_j}$$

$$= \sum_j v_j^2 p_j(1-p_j) - \sum_{j \neq k} v_j v_k p_j p_k$$

$$= \sum_j p_j v_j^2 - \left(\sum_j p_j v_j\right)^2$$

This is the variance of a random variable that takes value $v_j$ with probability $p_j$. Since variance is always non-negative, $\mathbf{v}^T H \mathbf{v} \geq 0$, proving that $g$ is convex.

## 2. Translation Invariance

I need to show that $g(\mathbf{x} + b) = g(\mathbf{x}) + b$, where $b$ is a scalar and $\mathbf{x} + b$ means adding $b$ to each element of $\mathbf{x}$.

$$g(\mathbf{x} + b) = \log \sum_i \exp(x_i + b) = \log \left(\exp(b) \sum_i \exp(x_i)\right)$$

Using the property $\log(ab) = \log(a) + \log(b)$:

$$g(\mathbf{x} + b) = \log(\exp(b)) + \log\left(\sum_i \exp(x_i)\right) = b + g(\mathbf{x})$$

This proves that $g$ is translation invariant in the sense that adding a constant to all inputs increases the output by that same constant.

## 3. Behavior with Very Large or Very Small Coordinates

### When some $x_i$ are very large:
If some coordinates are much larger than others, the sum will be dominated by these terms. If $x_m$ is the maximum value and significantly larger than other values:

$$g(\mathbf{x}) \approx \log(\exp(x_m)) = x_m$$

The function essentially returns the maximum value, with a small correction term for the contribution of other values.

### When all $x_i$ are very small:
If all values are very small (large negative numbers), the exponentials become nearly zero, which can lead to numerical underflow. The log of a very small number approaches negative infinity, making computation unstable.

## 4. Numerical Stability with $b = \max_i x_i$

Let's substitute $b = \max_i x_i$ into our translation invariance result:

$$g(\mathbf{x}) = g(\mathbf{x} - b + b) = g(\mathbf{x} - b) + b$$

Where $\mathbf{x} - b$ means subtracting the maximum value from each element.

This gives us:
$$g(\mathbf{x}) = \log \sum_i \exp(x_i - \max_j x_j) + \max_j x_j$$

The term $\exp(x_i - \max_j x_j)$ is always ≤ 1, since $x_i - \max_j x_j ≤ 0$ for all $i$. This prevents overflow.

Also, at least one term in the sum equals 1 (when $i$ corresponds to the maximum value), ensuring the sum is at least 1, preventing underflow.

This transformation preserves the mathematical value while avoiding the numerical instabilities that occur when calculating exponentials of very large positive or negative numbers, making it computationally robust.

The technique of subtracting the maximum value is commonly used in practical implementations of softmax and log-sum-exp operations.

the first one is already proved previously 
for the second one I got G(x+b) = G(x) + b by using the property log(ab) = log(a) + log(b) 
For third, for such values the log becomes undefined. [WRONG}

7. Assume that we have some probability distribution $P$. Suppose we pick another distribution $Q$ with $Q(i) \propto P(i)^\alpha$ for $\alpha > 0$.
    1. Which choice of $\alpha$ corresponds to doubling the temperature? Which choice corresponds to halving it?
    1. What happens if we let the temperature approach $0$?
    1. What happens if we let the temperature approach $\infty$?


Write the equation and solve it 

# Temperature Effects in Probability Distributions

## 1. Relationship Between α and Temperature

In statistical physics and machine learning, the concept of temperature ($T$) relates to probability distributions through the Boltzmann distribution. When we define $Q(i) \propto P(i)^\alpha$, we're effectively creating a new distribution by raising the original probabilities to a power.

The standard relationship between temperature and probabilities is:

$$P(i) \propto \exp\left(-\frac{E_i}{kT}\right)$$

Where $E_i$ is energy, $k$ is Boltzmann's constant, and $T$ is temperature.

If we define $P(i) \propto \exp\left(-\frac{E_i}{T}\right)$ (absorbing the constant $k$ into $T$), then:

$$P(i)^\alpha \propto \exp\left(-\alpha \cdot \frac{E_i}{T}\right) = \exp\left(-\frac{E_i}{T/\alpha}\right)$$

This means $Q(i) \propto P(i)^\alpha$ corresponds to a new temperature $T' = T/\alpha$.

Therefore:
- **Doubling the temperature** ($T' = 2T$) corresponds to $\alpha = 1/2$
- **Halving the temperature** ($T' = T/2$) corresponds to $\alpha = 2$

Intuitively:
- Higher temperature (smaller $\alpha$) makes the distribution more uniform
- Lower temperature (larger $\alpha$) makes the distribution more concentrated on the highest probability events

## 2. Temperature Approaching Zero (α → ∞)

As $\alpha \to \infty$ (temperature approaches 0), we get:

$$Q(i) \propto P(i)^\alpha$$

For any probability values less than 1, raising them to a very large power approaches 0. If $P(i) < P(j)$, then as $\alpha$ grows, $P(i)^\alpha/P(j)^\alpha$ approaches 0.

After normalization, this means:
- The most likely state(s) under $P$ will have probability approaching 1 in $Q$
- All other states will have probability approaching 0 in $Q$

This is known as a "winner-takes-all" distribution—$Q$ becomes deterministic, placing all probability mass on the most likely state(s) according to $P$.

## 3. Temperature Approaching Infinity (α → 0)

As $\alpha \to 0$ (temperature approaches infinity), we get:

$$Q(i) \propto P(i)^\alpha$$

When $\alpha$ is very close to 0, $P(i)^\alpha$ approaches 1 for all $i$ with non-zero probability, regardless of the original probabilities.

After normalization:
- If there are $n$ states with non-zero probability in $P$, each state will have probability approximately $1/n$ in $Q$
- $Q$ approaches a uniform distribution over all states with non-zero probability in $P$

This makes intuitive sense: at infinite temperature, all energy states become equally likely, regardless of their energy differences.

## Practical Applications

This relationship between temperature and distributions has important applications:
- In simulated annealing, we gradually lower the temperature to help find global optima
- In machine learning, temperature scaling is used to control the "sharpness" of predicted probabilities
- In reinforcement learning, temperature controls exploration vs. exploitation

The parameter $\alpha$ gives us a simple way to control how concentrated or uniform our distribution becomes, which is valuable in many optimization and learning contexts.